# OpenSearch database explorer

This notebook is independent of the application code. It connects directly to OpenSearch and helps inspect:

- cluster and index status
- index mappings and stored fields
- document counts and sample documents
- metadata, titles, sources, and chunk sizes
- simple charts describing the indexed data

Vector embeddings are excluded from displayed documents because they are large numeric arrays.

In [1]:
import json
import os
from collections import Counter
from getpass import getpass

import altair as alt
import pandas as pd
from IPython.display import display
from opensearchpy import OpenSearch, helpers

alt.data_transformers.disable_max_rows()

# Change these values here, or set the corresponding environment variables.
OPENSEARCH_URL = os.getenv("OPENSEARCH_URL", "http://localhost:9200")
INDEX_NAME = os.getenv("OPENSEARCH_INDEX", "mickey_mouse_wiki_articles_v1")
USERNAME = os.getenv("OPENSEARCH_USERNAME", "admin")
PASSWORD = os.getenv("OPENSEARCH_PASSWORD") or getpass("OpenSearch password: ")

# Limit the number of documents loaded into pandas for visualization.
# The exact index count is still retrieved separately.
MAX_DOCUMENTS_FOR_ANALYSIS = int(os.getenv("OPENSEARCH_VIS_MAX_DOCS", "10000"))

client = OpenSearch(
hosts=[OPENSEARCH_URL],
http_auth=(USERNAME, PASSWORD),
use_ssl=OPENSEARCH_URL.startswith("https://"),
verify_certs=False,
ssl_show_warn=False,
)

print(f"OpenSearch: {OPENSEARCH_URL}")
print(f"Index: {INDEX_NAME}")
print(f"Analysis limit: {MAX_DOCUMENTS_FOR_ANALYSIS:,} documents")

OpenSearch: http://localhost:9200
Index: mickey_mouse_wiki_articles_v1
Analysis limit: 10,000 documents


In [2]:
DOC_INDEX_NAME = os.getenv("OPENSEARCH_DOC_INDEX", "mickey_mouse_wiki_articles_docs_v1")
DOC_INDEX_SAMPLE_SIZE = 100

if not client.indices.exists(index=DOC_INDEX_NAME):
    raise RuntimeError(f"Document index '{DOC_INDEX_NAME}' does not exist.")

doc_hits = client.search(
    index=DOC_INDEX_NAME,
    body={
        "size": DOC_INDEX_SAMPLE_SIZE,
        "query": {"match_all": {}},
        "_source": ["doc_id", "revision_id", "document"],
    },
).get("hits", {}).get("hits", [])

doc_rows = []
for hit in doc_hits:
    src = hit.get("_source", {})
    doc = src.get("document", {})
    metadata = doc.get("metadata", {}) or {}
    page_content = str(doc.get("page_content", "") or "")
    doc_rows.append(
        {
            "_id": hit.get("_id"),
            "doc_id": src.get("doc_id"),
            "revision_id": src.get("revision_id"),
            "text_preview": page_content[:200],
            "content_chars": len(page_content),
        }
    )

doc_index_df = pd.DataFrame(doc_rows)
print(f"Showing {len(doc_index_df)} documents from '{DOC_INDEX_NAME}'")
display(doc_index_df)

Showing 31 documents from 'mickey_mouse_wiki_articles_docs_v1'


,_id,doc_id,revision_id,text_preview,content_chars
0,20859,20859,1363833276,Disney cartoon character and mascot\n\nFor oth...,233511
1,78136113,78136113,1363592787,American animated interactive children's telev...,186898
2,2300535,2300535,1363593716,American animated interactive children's telev...,169864
3,3825974,3825974,1357530419,"Short film series\n\n""List of Mickey Mouse car...",229875
4,77553,77553,1361933987,Fictional universe involving Mickey Mouse and ...,208176
5,25280564,25280564,1363393874,This is a full list of episodes from the Playh...,111405
6,20664,20664,1362371245,American variety television show\n\nNot to be ...,83994
7,49875691,49875691,1362193591,American animated children's TV series\n\n| *M...,172333
8,67664922,67664922,1362013697,American animated children's television series...,174379
9,38797624,38797624,1356664983,American animated television series\n\n| *Mick...,184529


## 1. Check the connection

In [3]:
cluster_info = client.info()
print(f"Cluster: {cluster_info.get('cluster_name', 'unknown')}")
print(f"Version: {cluster_info.get('version', {}).get('number', 'unknown')}")
print(f"Node: {cluster_info.get('name', 'unknown')}")

if not client.indices.exists(index=INDEX_NAME):
    raise RuntimeError(f"Index '{INDEX_NAME}' does not exist.")

print(f"Index '{INDEX_NAME}' is available.")

Cluster: docker-cluster
Version: 2.13.0
Node: 2a566af8218f
Index 'mickey_mouse_wiki_articles_v1' is available.


## 2. Index health and storage overview

In [6]:
health = client.cluster.health(index=INDEX_NAME)
stats = client.indices.stats(index=INDEX_NAME)
index_stats = stats["indices"][INDEX_NAME]
primary_stats = index_stats.get("primaries", {})

overview = pd.DataFrame([
    {"metric": "health", "value": health.get("status")},
    {"metric": "documents", "value": primary_stats.get("docs", {}).get("count")},
    {"metric": "deleted documents", "value": primary_stats.get("docs", {}).get("deleted")},
    {"metric": "index size", "value": primary_stats.get("store", {}).get("size_in_bytes")},
    {"metric": "shards", "value": health.get("active_shards")},
])
display(overview)

print("\nIndex statistics in JSON:")
display(pd.json_normalize(index_stats, sep="."))

,metric,value
0,health,yellow
1,documents,643
2,deleted documents,14
3,index size,21082021
4,shards,1



Index statistics in JSON:


,uuid,primaries.docs.count,primaries.docs.deleted,primaries.store.size_in_bytes,primaries.store.reserved_in_bytes,primaries.indexing.index_total,primaries.indexing.index_time_in_millis,primaries.indexing.index_current,primaries.indexing.index_failed,primaries.indexing.delete_total,...,total.translog.remote_store.upload.total_upload_size.started_bytes,total.translog.remote_store.upload.total_upload_size.failed_bytes,total.translog.remote_store.upload.total_upload_size.succeeded_bytes,total.request_cache.memory_size_in_bytes,total.request_cache.evictions,total.request_cache.hit_count,total.request_cache.miss_count,total.recovery.current_as_source,total.recovery.current_as_target,total.recovery.throttle_time_in_millis
0,jlC3MR1WRuyiNdVylen5ig,643,14,21082021,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Inspect the mapping

The mapping shows how OpenSearch stores searchable text, metadata, and vector fields.

In [26]:
mapping_response = client.indices.get_mapping(index=INDEX_NAME)
index_mapping = mapping_response[INDEX_NAME].get("mappings", {})
print(json.dumps(index_mapping, indent=2))

{
  "properties": {
    "doc_id": {
      "type": "keyword"
    },
    "document": {
      "properties": {
        "metadata": {
          "dynamic": "true",
          "properties": {
            "page_url": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                  "ignore_above": 256
                }
              }
            },
            "pageid": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                  "ignore_above": 256
                }
              }
            },
            "revision_id": {
              "type": "long"
            },
            "source": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                  "ignore_above": 256
                }
              }
            },
            "title": {
              "type

## 4. Count and preview stored documents

The vector field is excluded from the preview so the output stays readable.

In [7]:
total_documents = client.count(index=INDEX_NAME)["count"]
print(f"Total documents in '{INDEX_NAME}': {total_documents:,}")

preview_response = client.search(
    index=INDEX_NAME,
    body={
        "size": min(10, total_documents),
        "_source": {"excludes": ["embedding", "vector"]},
        "query": {"match_all": {}},
    },
)

preview_rows = []
for hit in preview_response["hits"]["hits"]:
    source = hit.get("_source", {})
    metadata = source.get("metadata", {}) or {}
    preview_rows.append({
        "_id": hit.get("_id"),
        "title": metadata.get("title"),
        "pageid": metadata.get("pageid"),
        "chunk_id": metadata.get("chunk_id"),
        "text_preview": str(source.get("text", ""))[:300],
        "metadata": metadata,
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)
if preview_rows:
    print("\nComplete first document (without vector data):")
    display(preview_rows[0])

Total documents in 'mickey_mouse_wiki_articles_v1': 643


,_id,title,pageid,chunk_id,text_preview,metadata
0,a877d066-05e1-4374-a723-1b258117756e,Mickey Mouse,20859,20859_0,Disney cartoon character and mascot \nFor oth...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
1,e5ea0c99-3566-4966-8571-2383175ad739,Mickey Mouse,20859,20859_1,* [1 Creation](#Creation)\n+ [1.1 Design](#Des...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
2,6c34e729-20f9-41f7-baf7-8536473879f8,Mickey Mouse,20859,20859_2,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
3,7dde6133-65e6-4513-83a8-8f4e19548961,Mickey Mouse,20859,20859_3,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
4,b2516ccd-3641-41ec-acdc-2d936da55cb6,Mickey Mouse,20859,20859_4,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
5,eee0657a-6c0c-4d75-8b5b-4df286217131,Mickey Mouse,20859,20859_5,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
6,66f0248d-988c-4283-acee-1751c46fd93f,Mickey Mouse,20859,20859_6,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
7,9a009cf7-1cd8-4975-b77c-d2fbb436639c,Mickey Mouse,20859,20859_7,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
8,33614e9d-1dab-40da-9327-6eeb8b0da44c,Mickey Mouse,20859,20859_8,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
9,b8b333d8-d4e1-455e-ba75-b2ef88ba811c,Mickey Mouse,20859,20859_9,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...



Complete first document (without vector data):


{'_id': 'a877d066-05e1-4374-a723-1b258117756e',
 'title': 'Mickey Mouse',
 'pageid': '20859',
 'chunk_id': '20859_0',
 'text_preview': 'Disney cartoon character and mascot  \nFor other uses, see [Mickey Mouse (disambiguation)](/wiki/Mickey_Mouse_(disambiguation) "Mickey Mouse (disambiguation)").  \nFictional character  \n| Mickey Mouse | |\n| --- | --- |\n| *[Mickey Mouse & Friends](/wiki/Mickey_Mouse_universe "Mickey Mouse universe")* c',
 'metadata': {'page_url': 'https://en.wikipedia.org/wiki/Mickey_Mouse',
  'source': 'https://en.wikipedia.org/wiki/Mickey_Mouse',
  'title': 'Mickey Mouse',
  'pageid': '20859',
  'chunk_id': '20859_0',
  'revision_id': 1363833276}}

## 5. Load documents for analysis

This uses the OpenSearch scroll helper and excludes embeddings. Increase `MAX_DOCUMENTS_FOR_ANALYSIS` if the index is larger and more data is needed.

In [28]:
records = []
scan_query = {
    "_source": {"excludes": ["embedding", "vector"]},
    "query": {"match_all": {}},
}

for hit in helpers.scan(
    client,
    index=INDEX_NAME,
    query=scan_query,
    size=500,
    scroll="2m",
    preserve_order=False,
):
    source = hit.get("_source", {})
    metadata = source.get("metadata", {}) or {}
    text = str(source.get("text", source.get("page_content", "")) or "")
    records.append({
        "_id": hit.get("_id"),
        "text": text,
        "text_length": len(text),
        "word_count": len(text.split()),
        "title": metadata.get("title", "Unknown"),
        "pageid": metadata.get("pageid"),
        "source": metadata.get("source", "Unknown"),
        "page_url": metadata.get("page_url", "Unknown"),
        "revision_id": metadata.get("revision_id"),
        "chunk_id": metadata.get("chunk_id"),
        "metadata": metadata,
    })
    if len(records) >= MAX_DOCUMENTS_FOR_ANALYSIS:
        break

documents_df = pd.DataFrame(records)
print(f"Loaded {len(documents_df):,} documents for analysis.")
if len(documents_df) < total_documents:
    print(f"Analysis is capped at {MAX_DOCUMENTS_FOR_ANALYSIS:,}; index count remains {total_documents:,}.")
display(documents_df.head(100))

Loaded 25 documents for analysis.


,_id,text,text_length,word_count,title,pageid,source,page_url,revision_id,chunk_id,metadata
0,20859,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
1,78136113,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
2,2300535,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
3,3825974,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
4,77553,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
5,25280564,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
6,20664,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
7,49875691,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
8,38797624,,0,0,Unknown,None,Unknown,Unknown,None,None,{}
9,67664922,,0,0,Unknown,None,Unknown,Unknown,None,None,{}


## 6. Metadata summary

In [15]:
if documents_df.empty:
    print("No documents were returned.")
else:
    metadata_keys = Counter(
        key
        for metadata in documents_df["metadata"]
        for key in metadata
    )
    metadata_summary = pd.DataFrame(
        metadata_keys.most_common(),
        columns=["metadata_field", "document_count"],
    )
    print("Metadata fields:")
    display(metadata_summary)

    print("Unique values:")
    unique_summary = pd.DataFrame([
        {"field": field, "unique_values": documents_df[field].nunique(dropna=True)}
        for field in ["title", "pageid", "source", "revision_id"]
        if field in documents_df
    ])
    display(unique_summary)

Metadata fields:


,metadata_field,document_count
0,page_url,579
1,source,579
2,title,579
3,pageid,579
4,chunk_id,579
5,revision_id,579


Unique values:


,field,unique_values
0,title,25
1,pageid,25
2,source,25
3,revision_id,25


## 7. Documents and chunks by Wikipedia page

Because each Markdown chunk is stored as one OpenSearch document, repeated titles show how many chunks were created for each page.

In [16]:
if not documents_df.empty:
    title_counts = (
        documents_df["title"]
        .fillna("Unknown")
        .value_counts()
        .head(20)
        .rename_axis("title")
        .reset_index(name="chunks")
    )
    display(title_counts)

    title_chart = (
        alt.Chart(title_counts)
        .mark_bar()
        .encode(
            x=alt.X("chunks:Q", title="Chunk count"),
            y=alt.Y("title:N", sort="-x", title="Wikipedia page"),
            tooltip=["title", "chunks"],
        )
        .properties(title="Top pages by chunk count", height=450)
    )
    display(title_chart)

,title,chunks
0,Mickey Mouse universe,92
1,The Mickey Mouse Club,38
2,Mickey Mouse (film series),37
3,Donald Duck,37
4,Mickey Mouse,34
5,Pete (Disney),32
6,Disneyland,31
7,List of Mickey Mouse appearances in Disney media,24
8,Mickey Mouse Mixed-Up Adventures,23
9,Mickey Mouse Clubhouse,21


alt.Chart(...)

## 8. Chunk size distribution

In [17]:
if not documents_df.empty:
    size_summary = documents_df[["text_length", "word_count"]].describe().round(2)
    display(size_summary)

    length_chart = (
        alt.Chart(documents_df)
        .mark_bar()
        .encode(
            x=alt.X("text_length:Q", bin=alt.Bin(maxbins=30), title="Characters per chunk"),
            y=alt.Y("count():Q", title="Number of chunks"),
            tooltip=[alt.Tooltip("count():Q", title="Chunks")],
        )
        .properties(title="Chunk character-length distribution", width=700)
    )
    display(length_chart)

,text_length,word_count
count,579.00,579.00
mean,5325.15,575.78
std,15938.33,1596.80
min,84.00,4.00
25%,706.00,82.00
50%,1484.00,174.00
75%,3992.50,469.50
max,138857.00,14098.00


alt.Chart(...)

## 9. Source distribution

In [18]:
if not documents_df.empty:
    source_counts = (
        documents_df["source"]
        .fillna("Unknown")
        .value_counts()
        .head(15)
        .rename_axis("source")
        .reset_index(name="chunks")
    )
    display(source_counts)

    source_chart = (
        alt.Chart(source_counts)
        .mark_bar()
        .encode(
            x=alt.X("chunks:Q", title="Chunk count"),
            y=alt.Y("source:N", sort="-x", title="Source"),
            tooltip=["source", "chunks"],
        )
        .properties(title="Top sources by chunk count", height=350)
    )
    display(source_chart)

,source,chunks
0,https://en.wikipedia.org/wiki/Mickey_Mouse_uni...,92
1,https://en.wikipedia.org/wiki/The_Mickey_Mouse...,38
2,https://en.wikipedia.org/wiki/Mickey_Mouse_(fi...,37
3,https://en.wikipedia.org/wiki/Donald_Duck,37
4,https://en.wikipedia.org/wiki/Mickey_Mouse,34
5,https://en.wikipedia.org/wiki/Pete_(Disney),32
6,https://en.wikipedia.org/wiki/Disneyland,31
7,https://en.wikipedia.org/wiki/List_of_Mickey_M...,24
8,https://en.wikipedia.org/wiki/Mickey_Mouse_Mix...,23
9,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,21


alt.Chart(...)

## ALERT ALERT !!!!! ------ DELETE BELOW ------ ALERT ALERT !!!!!

In [2]:
# Permanently delete the configured OpenSearch index.
if client.indices.exists(index=INDEX_NAME):
    response = client.indices.delete(index=INDEX_NAME)
    print(f"Deleted index '{INDEX_NAME}': {response.get('acknowledged', False)}")
else:
    print(f"Index '{INDEX_NAME}' does not exist.")

Deleted index 'mickey_mouse_wiki_articles_docs_v1': True


In [26]:
# Check if a specific document exists in the index
TARGET_INDEX = "mickey_mouse_wiki_articles_docs_v1"
DOC_ID = "2280793"  # change to any doc ID you want to check

if client.indices.exists(index=TARGET_INDEX):
    exists = client.exists(index=TARGET_INDEX, id=DOC_ID)
    if exists:
        print(f"﷿﷿﷿ Document '{DOC_ID}' EXISTS in index '{TARGET_INDEX}'.")
        doc = client.get(
            index=TARGET_INDEX,
            id=DOC_ID
        )
        src = doc["_source"]
        print(f"   Doc ID    : {src.get('doc_id', 'N/A')}")
        print(f"   Revision ID : {src.get('revision_id', 'N/A')}")
    else:
        print(f"﷿﷿﷿ Document '{DOC_ID}' does NOT exist in index '{TARGET_INDEX}'.")
else:
    print(f"﷿﷿﷿﷿﷿﷿  Index '{TARGET_INDEX}' does not exist.")

﷿﷿﷿ Document '2280793' does NOT exist in index 'mickey_mouse_wiki_articles_docs_v1'.
